# 02 — Delta Ingestion Pipeline

This notebook demonstrates the full DeltaIngestor pipeline: manifest comparison, skip-or-ingest decision, and Neptune graph write.

**What you'll learn:**
- How ManifestManager tracks CPG state in S3
- How DeltaIngestor decides to skip or ingest
- How tenant-based graph replacement works

**Prerequisites:**
- Neptune Analytics or Neptune Database endpoint
- S3 bucket for manifests
- IAM permissions for Neptune and S3

In [ ]:
# %pip install codeproperty-graph graphrag-toolkit-document-graph[graphrag]

In [ ]:
import json
import asyncio
from codeproperty_graph import DeltaIngestor, ManifestManager, GraphDiff, CPGNode

## Configuration

In [ ]:
# Replace with your values:
BUCKET = "my-graphrag-artifacts"  # S3 bucket for manifests
REPO = "example-service"          # Repository name
JOB_ID = "build-001"             # CI/CD build identifier
NEPTUNE_ENDPOINT = "neptune-analytics://my-graph-id"  # or neptune-db://...

# Tenant ID (auto-generated in production, fixed here for demo)
TENANT_ID = "demo123"

## Load Joern Export

In [ ]:
import json
from graphrag_toolkit.codeproperty_graph import CPGNode, CPGEdge, NodeType

with open('data/sample_nodes.json') as f:
    nodes = [CPGNode.from_joern(raw) for raw in json.load(f)]

with open('data/sample_edges.json') as f:
    edges = [CPGEdge.from_joern(raw) for raw in json.load(f)]

print(f'Parsed {len(nodes)} nodes ({len(set(n.node_type for n in nodes))} types)')
print(f'Parsed {len(edges)} edges ({len(set(e.edge_type for e in edges))} types)')


## Step 1: Extract Method Signatures

Only METHOD nodes participate in change detection.

In [ ]:
# Only METHOD nodes participate in delta comparison
methods = [n for n in nodes if n.node_type == NodeType.METHOD]
current_sigs = {m.full_name: m.hash for m in methods}

print(f'Extracted {len(current_sigs)} method signatures (from {len(nodes)} total nodes):')
for name, h in current_sigs.items():
    print(f'  {name}: {h}')

print(f'\nOther node types (not in delta): {set(n.node_type for n in nodes if n.node_type != NodeType.METHOD)}')


## Step 2: Check Previous Manifest

ManifestManager retrieves the last known state from S3.

In [ ]:
manifest_mgr = ManifestManager(bucket=BUCKET)

# In production this reads from S3. For demo, simulate:
# previous_manifest = manifest_mgr.get(REPO)

# Simulate: first run (no previous manifest)
previous_manifest = None

if previous_manifest is None:
    print("No previous manifest — this is a fresh ingestion.")
    print("All methods will be treated as ADDED.")
else:
    print(f"Previous manifest: tenant={previous_manifest.tenant_id}, job={previous_manifest.job_id}")

## Step 3: Compute Diff

In [ ]:
previous_sigs = previous_manifest.method_signatures if previous_manifest else {}

diff = GraphDiff.compare(previous_sigs, current_sigs)
print(f"Diff: {diff.summary}")
print(f"Has changes: {diff.has_changes}")

if diff.has_changes:
    print("\n→ Will INGEST to Neptune")
else:
    print("\n→ Will SKIP (no Neptune writes)")

## Step 4: Write to Neptune (if changed)

In production, this uses the GraphStore. For this demo, we simulate the write.

In [ ]:
# Simulated write function (replace with actual Neptune write in production)
write_log = []

async def mock_write_fn(query: str, params: dict):
    """Simulates a Neptune write — logs instead of executing."""
    write_log.append({'query': query[:80], 'param_count': len(params)})

if diff.has_changes:
    print(f"Writing {len(nodes_data)} nodes and {len(edges_data)} edges...")
    print(f"Tenant: {TENANT_ID}")
    print(f"(Using mock write — replace with graph_store.execute_query in production)")
    
    # In production:
    # result = await ingestor.ingest(repo=REPO, job_id=JOB_ID, ...)
    
    # Simulated result:
    result = {
        "status": "INGESTED",
        "delta": diff.summary,
        "tenant_id": TENANT_ID,
        "nodes_written": len(nodes_data),
        "edges_written": len(edges_data),
    }
    print(f"\nResult: {json.dumps(result, indent=2)}")
else:
    result = {"status": "SKIPPED", "reason": "no changes detected"}
    print(f"Result: {json.dumps(result, indent=2)}")

## Step 5: Simulate Second Run (No Changes)

On the next CI/CD run with no code changes, the ingestor skips entirely.

In [ ]:
# Same signatures as before — simulates "no code change" commit
second_run_sigs = current_sigs.copy()

diff2 = GraphDiff.compare(current_sigs, second_run_sigs)
print(f"Second run diff: {diff2.summary}")
print(f"Has changes: {diff2.has_changes}")
print()
print("→ SKIPPED — zero Neptune writes, zero cost, <1 second")
print("  This is the common case in CI/CD (most commits don't change method bodies)")

## Summary

| Scenario | Action | Neptune Cost | Time |
|----------|--------|-------------|------|
| First run (no manifest) | Full ingest | N nodes + M edges | Seconds |
| Code changed | Full replace (new tenant, purge old) | N nodes + M edges + purge | Seconds |
| No code change | SKIP | Zero | <1s |

The delta logic ensures you only pay for Neptune writes when code actually changes.